<a href="https://colab.research.google.com/github/rupanshi1627-create/Autonomous-agent-for-test-automation-project02/blob/main/Autonomus_agent_for_test_automation_project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
!pip install -q langgraph
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
#pip is python app store all the libraries comes form here
#LangGraph is an open-source framework from LangChain designed to build and manage AI agent workflows using graph-based structures.

In [26]:
%%writefile spec.md
# Calculator module requirements

1. add(a, b) returns the sum of a and b.
2. divide(a, b) returns a divided by b. Dividing by zero must raise ValueError.
3. is_even(n) returns True if n is even, otherwise False.

Overwriting spec.md


In [27]:
!cat spec.md

# Calculator module requirements

1. add(a, b) returns the sum of a and b.
2. divide(a, b) returns a divided by b. Dividing by zero must raise ValueError.
3. is_even(n) returns True if n is even, otherwise False.


In [28]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class AgentState(TypedDict):
    requirements: str
    test_cases: str
    summary: str


def call_llm(prompt: str) -> str:
    return f"[stub LLM reply to a prompt of {len(prompt)} characters]"

In [29]:
def read_requirements(state: AgentState) -> dict:
    with open("spec.md", encoding="utf-8") as f:
        return {"requirements": f.read()}


def generate_tests(state: AgentState) -> dict:
    prompt = f"Write pytest test cases for these requirements:\n{state['requirements']}"
    return {"test_cases": call_llm(prompt)}


def summarize(state: AgentState) -> dict:
    prompt = f"Summarize these test cases in 2 lines:\n{state['test_cases']}"
    return {"summary": call_llm(prompt)}

In [30]:
builder = StateGraph(AgentState)
builder.add_node("read_requirements", read_requirements)
builder.add_node("generate_tests", generate_tests)
builder.add_node("summarize", summarize)

builder.add_edge(START, "read_requirements")
builder.add_edge("read_requirements", "generate_tests")
builder.add_edge("generate_tests", "summarize")
builder.add_edge("summarize", END)

graph = builder.compile()

In [32]:
final_state = graph.invoke({})
for key, value in final_state.items():
    print(f"--- {key} ---")
    print(value)

--- requirements ---
# Calculator module requirements

1. add(a, b) returns the sum of a and b.
2. divide(a, b) returns a divided by b. Dividing by zero must raise ValueError.
3. is_even(n) returns True if n is even, otherwise False.

--- test_cases ---
[stub LLM reply to a prompt of 261 characters]
--- summary ---
[stub LLM reply to a prompt of 85 characters]


In [34]:
%%writefile calculator.py
def add(a, b):
    return int(a) + int(b)


def divide(a, b):
    return a / b


def is_even(n):
    return n % 2 == 0 and n != 0

Overwriting calculator.py


In [36]:
%%writefile test_calculator.py
import pytest
from calculator import add, divide, is_even


def test_add_integers():
    assert add(2, 3) == 5


def test_add_floats():
    assert add(1.5, 2.5) == 4.0


def test_divide_normal():
    assert divide(10, 2) == 5


def test_divide_by_zero():
    with pytest.raises(ValueError):
        divide(1, 0)


def test_is_even_true():
    assert is_even(4) is True


def test_is_even_zero():
    assert is_even(0) is True


def test_is_even_false():
    assert is_even(3) is False

Overwriting test_calculator.py


In [39]:
!python -m pytest -q --tb=short test_calculator.py

.F.F.F.                                                                  [100%]
=================================== FAILURES ===================================
_______________________________ test_add_floats ________________________________
test_calculator.py:10: in test_add_floats
    assert add(1.5, 2.5) == 4.0
E   assert 3 == 4.0
E    +  where 3 = add(1.5, 2.5)
_____________________________ test_divide_by_zero ______________________________
test_calculator.py:19: in test_divide_by_zero
    divide(1, 0)
calculator.py:6: in divide
    return a / b
           ^^^^^
E   ZeroDivisionError: division by zero
______________________________ test_is_even_zero _______________________________
test_calculator.py:27: in test_is_even_zero
    assert is_even(0) is True
E   assert False is True
E    +  where False = is_even(0)
=========================== short test summary info ============================
FAILED test_calculator.py::test_add_floats - assert 3 == 4.0
FAILED test_calculator.py::test_

In [40]:
import subprocess
import sys


def run_pytest() -> str:
    """Tool: run the tests and return the text report."""
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "--tb=short", "test_calculator.py"],
        capture_output=True,
        text=True,
    )
    return result.stdout + result.stderr


print(run_pytest())

.F.F.F.                                                                  [100%]
=================================== FAILURES ===================================
_______________________________ test_add_floats ________________________________
test_calculator.py:10: in test_add_floats
    assert add(1.5, 2.5) == 4.0
E   assert 3 == 4.0
E    +  where 3 = add(1.5, 2.5)
_____________________________ test_divide_by_zero ______________________________
test_calculator.py:19: in test_divide_by_zero
    divide(1, 0)
calculator.py:6: in divide
    return a / b
           ^^^^^
E   ZeroDivisionError: division by zero
______________________________ test_is_even_zero _______________________________
test_calculator.py:27: in test_is_even_zero
    assert is_even(0) is True
E   assert False is True
E    +  where False = is_even(0)
=========================== short test summary info ============================
FAILED test_calculator.py::test_add_floats - assert 3 == 4.0
FAILED test_calculator.py::test_